# 18장 · SDK와 도구 실행을 복습하고 작은 과제 완성하기

도구 루프 결과에 대출 기간이 없습니다. 새 기술을 더하기 전에 입력 검사가 잘못된 요청까지 통과시킨 것은 아닌지 살펴봅니다. 17장에서 쓰던 대출 규칙 도구 한 건을 골라 원래 요청을 바꾸지 않는 검사 함수를 완성합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 실행 직전의 요청 한 건 읽기

get_loan_rule은 입력 칸이 없는 도구입니다. 요청에는 이름과 비어 있지 않은 호출 ID가 있어야 하고 arguments에는 빈 JSON 객체를 적은 문자열이 와야 합니다.



**볼 곳**　‘도구 요청’가 가운데 ‘인자 검사’에 도착한 뒤 오른쪽에 어떤 결과가 남는지 보세요.

**애니메이션의 의미**　실행 전에 도구 이름과 인자가 약속에 맞는지 봅니다.


### 셀 01 · 빈 객체와 빈 문자열 구분하기

겉보기에는 내용이 없어도 문자열을 풀었을 때 어떤 값이 남는지가 중요합니다.

In [1]:
import json
item = {"name": "get_loan_rule", "call_id": "c1", "arguments": "{}"}
print(json.loads(item["arguments"]) == {})

True


True가 나옵니다. 빈 문자열은 JSON 문서가 아니어서 같은 변환을 통과할 수 없습니다. []는 읽을 수 있는 JSON이지만 이 도구가 정한 객체 모양과 다릅니다.

### 셀 02 · 띄어쓰기만 있는 ID 확인하기

ID에 공백 세 칸만 들어온 경우를 따로 확인합니다.

In [2]:
call_id = "   "
print(bool(call_id.strip()))

False


False이므로 결과를 연결할 이름이 없습니다. strip을 빼면 원래 세 칸의 길이 때문에 유효하다고 잘못 판단할 수 있습니다.

## 검사 함수를 고쳐 한 건 통과시키기

요청의 바깥 자료형부터 확인하고 안쪽 칸을 읽습니다. 유효하지 않은 입력이면 함수 안에서 False를 돌려주면 됩니다. 사전의 값을 덮어쓰지 않으면 실패 원인을 나중에도 다시 읽을 수 있습니다.



**볼 곳**　‘검사 수정’를 기준으로 양쪽의 ‘실패한 입력’와 ‘같은 입력’를 대조하세요.

**애니메이션의 의미**　검사 함수를 고친 뒤 같은 입력을 다시 넣습니다.


#### 같은 c1 요청에서 어느 층을 바꿨나요?

<table style="width:100%;border-collapse:collapse"><thead><tr><th scope="col">원본에서 바꾼 부분</th><th scope="col">풀이 함수의 반환</th><th scope="col">이유</th></tr></thead><tbody><tr><td>원본 arguments는 문자열 &quot;{}&quot;</td><td>True</td><td>도구에 전달할 인자가 없음</td></tr><tr><td>arguments를 문자열 {&quot;day&quot;:&quot;weekend&quot;}로 변경</td><td>False</td><td>JSON을 푼 안쪽 객체에 인자가 생김</td></tr><tr><td>바깥 item에만 day를 추가하고 arguments는 &quot;{}&quot; 유지</td><td>True</td><td>풀이 함수는 바깥의 부가 필드를 읽지 않음</td></tr></tbody></table>

### 과제 03 · 요청 검사 함수 완성하기

사전, 도구 이름, 공백이 아닌 문자열 ID, 문자열로 온 빈 JSON 객체 순서로 검사하세요. 변환에 실패해도 예외 대신 False를 돌려줍니다.

get_loan_rule은 인자를 받지 않습니다. 요청 바깥에서 name과 call_id를 읽고, arguments 문자열을 풀어 빈 객체인지 확인하세요. day를 어느 쪽에 넣었는지에 따라 검사 결과가 달라집니다.

In [3]:
def validate_request(item):
    pass  # 위 조건을 차례로 검사하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def validate_request(item):
    if not isinstance(item, dict) or item.get(&quot;name&quot;) != &quot;get_loan_rule&quot;:
        return False
    cid = item.get(&quot;call_id&quot;)
    if not isinstance(cid, str) or not cid.strip():
        return False
    raw = item.get(&quot;arguments&quot;)
    if not isinstance(raw, str):
        return False
    try:
        return json.loads(raw) == {}
    except ValueError:
        return False

print(validate_request(item))</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>원래 c1 요청은 True입니다. 먼저 자료형을 검사해 None이나 목록이 들어와도 .get을 잘못 호출하는 오류를 피할 수 있습니다. 문자열로 온 {}만 허용하므로 파이썬 사전 자체를 넣은 요청도 거부합니다.</p>
</details>

## 조회 결과가 없는 기록 설명하기

이제 증상을 보고 필요한 부분으로 돌아갑니다. 인자를 풀지 못한 경우라면 15장의 arguments를, 요청 ID와 결과 ID가 다르면 16장 왕복을, 상한이 보이면 17장 마지막 반복을 다시 살펴보세요.



**볼 곳**　‘조회 결과’에서 갈라지는 두 경우를 비교하세요. 위쪽은 ‘기록 있음’, 아래쪽은 ‘기록 없음’입니다.

**애니메이션의 의미**　결과가 비었다면 기록을 찾지 못한 이유를 확인합니다.


### 셀 04 · 실행 전 차단한 요청 남기기

도구 이름이 허용 목록과 다른 요청을 받았습니다. 프로그램이 기록할 값 한 개를 계산합니다.

In [4]:
bad = {**item, "name": "other_lookup"}
allowed = bad["name"] == "get_loan_rule"
print(allowed)

False


False는 도구 선택 검사 결과입니다. 대출 기간이 0일이거나 데이터가 비어 있다는 뜻으로 바꿔 적으면 안 됩니다.

### 과제 05 · 동료에게 실패 원인 설명하기

요청 ID c1과 실패한 필드 이름을 한 문장으로 연결하세요. 실제 실행 결과를 만들지 마세요.

In [5]:
# bad의 call_id와 name을 읽어 진단 문구를 작성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(f&quot;{bad[&#x27;call_id&#x27;]} 요청은 도구 이름 {bad[&#x27;name&#x27;]}이 허용 목록에 없어 실행 전에 멈췄습니다.&quot;)</code></pre>
<p>실행 결과</p><pre><code>c1 요청은 도구 이름 other_lookup이 허용 목록에 없어 실행 전에 멈췄습니다.</code></pre>
<p>다른 사람이 c1의 원본 요청으로 돌아가 name을 고칠 수 있습니다. 이름을 고친 뒤에는 arguments와 call_id도 다시 확인하세요.</p>
</details>

## 돌아보기

기본 과제에서 잘못된 요청이 도구 실행 전에 멈췄는지 확인하세요. 더 연습하려면 보충 자료의 검사 중 한 건만 골라 입력, 멈춘 단계, 도구 호출 여부를 적으세요. 보충 자료에 나온 14건은 서로 다른 사례를 찾아볼 때 씁니다.